### import dependencies

In [ ]:
import openai
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery, Filter, FieldCondition, MatchAny
import tiktoken
import numpy as np
import json


### Create Qdrant Collection for Hybrid Search

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(
            size=1536,
            distance=Distance.COSINE
        )
    }
)

In [ ]:
# multiple queries against parent asin field
# we have one where we retrieve additional info price, image url
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_type=PayloadSchemaType.KEYWORD
)

### Embedding functions 

In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(
            input=text_list,
            model=model
        )
        return [item.embedding for item in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(
            input=batch,
            model=model 
        )
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    return all_embeddings

In [ ]:
df_reviews = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)
print(len(df_reviews))
df_reviews.head()

In [ ]:
def count_tokens(row):
    encoding = tiktoken.encoding_for_model("text-embedding-3-small")
    return len(encoding.encode(row["preprocessed_data"]))
    

In [ ]:
def preprocess_reviews_data(row):
    return f"{row['title']}: {row['text']}"

df_reviews["preprocessed_data"] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)


In [ ]:
df_reviews.head()

In [ ]:
total_tokens = df_reviews["token_count"].sum()
print(len(df_reviews))
print(f"Total tokens: {total_tokens}")
df_reviews = df_reviews[df_reviews["token_count"] <= 8192]
print(len(df_reviews))

### Embed the text and add additional fields to the payload of each vector for reviews

In [ ]:
df_data_to_embed = df_reviews[['preprocessed_data', 'parent_asin']]
df_data_to_embed.head()
data_to_embed_reviews = df_data_to_embed.to_dict(orient="records")


In [ ]:
print(data_to_embed_reviews[0])
print(len(data_to_embed_reviews))



In [ ]:
# data_to_embed_reviews
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [ ]:
text_to_embed_reviews

In [ ]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

In [ ]:
print(data_to_embed_reviews[0])

In [ ]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(PointStruct(
        id=i,
        vector={
            "text-embedding-3-small": embedding
        },
        payload=data
    ))
    i += 1

In [ ]:
pointstructs[0]

In [ ]:
batch_size = 100
counter = 1
for i in range(0, len(pointstructs), batch_size):
    batch = pointstructs[i:i+batch_size]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch
    )
    print(f"Upserted {min(i+batch_size, len(pointstructs))} of {len(pointstructs)}")

### A function to run search against reviews on a prefiltered set of product IDS

In [ ]:
def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_prefiltered_reviews_data(query, parent_asins, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    
    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    
    return results

In [ ]:
reviews = retrieve_prefiltered_reviews_data("bad quality", parent_asins=["B09Q5TNDHY"])



In [ ]:
reviews.points

In [ ]:
reviews = retrieve_prefiltered_reviews_data("bad quality", parent_asins=["B09Q5TNDHY", "BOB4NJ8NKN"])



In [ ]:
reviews.points

### Define the reviews retrieval tool

In [ ]:
def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_prefiltered_reviews_data(query, parent_asins, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    
    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

In [ ]:

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    
    return response.data[0].embedding


def retrieve_prefiltered_reviews_data(query, parent_asins, top_k=5): # 5 most similar items to users query
    qdrant_client = QdrantClient(url="http://localhost:6333") # connecting from within docker network
    embedding = get_embedding(query) # so we are actually creating related vector here
    
    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=top_k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_data", ""))
        similarity_scores.append(result.score)
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, user review: {chunk}\n"
    return formatted_context   


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:
    """
Get the top k reviews matching a query for a list of prefiltered items.

Args:
    query: The query to get the top k reviews for
    item_list: The list of item IDs to prefilter for before running the query
    top_k: The number of reviews to retrieve, this should be at least 20 if multiple items are prefiltered

Returns:
    A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for
"""
    retrievedContext = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        top_k=top_k
    )
    # if we set rerank to True, we re-order the retrieved context based on the reranker results
    formatted_context = process_context(retrievedContext)
    return formatted_context

In [ ]:
result = get_formatted_reviews_context(
    query="bad quality",
    parent_asins=["B09Q5TNDHY", "BOB4NJ8NKN"],
    top_k=5
)

In [ ]:
print(result)